In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('full_image_data_feb_25.csv')
books = pd.read_csv('full_book_data_feb_25.csv')

In [ ]:
q = practicalSPARQL.stringify_SPARQL('elements_query_050824.sparql')    # select data from the ttl file as a dataframe
df = sparql.select_as_dataframe(q)

In [ ]:
q = practicalSPARQL.stringify_SPARQL('books_query.sparql')    # select data from the ttl file as a dataframe
books = sparql.select_as_dataframe(q)

In [ ]:
# Filter for rising and setting keywords
keywords = ['CK_Heliacal Rising and Setting', 'CK_Chronical Rising and Setting', 'CK_Cosmic Rising and Setting']
mask = df['cks'].apply(lambda x: any(keyword in x for keyword in keywords))
rising_setting_df = df[mask].copy()

print(f"Total images with rising/setting CKS: {rising_setting_df.shape[0]}")
print(f"Unique books: {rising_setting_df['book'].nunique()}")
print(f"Unique images: {rising_setting_df['images'].nunique()}")

In [ ]:
# Create a dataset with book-level aggregation
# Group by book to get image counts and CKS value information

book_image_cks = rising_setting_df.groupby('book').agg({
    'images': 'nunique',  # Number of unique images per book
    'year': 'first',      # Publication year
    'cks': lambda x: '; '.join(set(x))  # Unique CKS values
}).reset_index()

book_image_cks.columns = ['book', 'unique_images', 'year', 'cks_values']

# Count which types of rising/setting each book has
def count_cks_types(cks_str):
    count = 0
    if 'Heliacal' in cks_str:
        count += 1
    if 'Chronical' in cks_str:
        count += 1
    if 'Cosmic' in cks_str:
        count += 1
    return count

book_image_cks['num_cks_types'] = book_image_cks['cks_values'].apply(count_cks_types)

# Get book metadata (publisher location, etc.)
book_meta = rising_setting_df.groupby('book').agg({
    'place': 'first',
    'custom_identifier': 'first'
}).reset_index()
book_meta.columns = ['book', 'place', 'book_id']

# Merge the metadata
book_image_cks = book_image_cks.merge(book_meta, on='book', how='left')

print(book_image_cks.head(10))

In [ ]:
# Create scatter plot: Year vs Number of Images per Book
# Color by number of CKS types
# Size by number of images

fig, ax = plt.subplots(figsize=(14, 8))

scatter = ax.scatter(
    book_image_cks['year'], 
    book_image_cks['unique_images'],
    c=book_image_cks['num_cks_types'],  # Color by number of CKS types
    s=book_image_cks['unique_images'] * 50,  # Size by number of images
    alpha=0.6,
    cmap='viridis',
    edgecolors='black',
    linewidth=1
)

ax.set_xlabel('Publication Year', fontsize=12)
ax.set_ylabel('Number of Unique Images per Book', fontsize=12)
ax.set_title('Books with Rising and Setting Images (CKS Values)\nColor = Number of CKS Types, Size = Number of Images', fontsize=14)
ax.grid(True, alpha=0.3)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Number of CKS Types', fontsize=11)
cbar.set_ticks([1, 2, 3])

plt.tight_layout()
plt.show()

In [ ]:
# Alternative scatter plot: Group by place (publishing location)
# Year vs Number of Images, colored by place

fig, ax = plt.subplots(figsize=(14, 8))

# Get unique places
places = book_image_cks['place'].unique()
colors = plt.cm.Set3(np.linspace(0, 1, len(places)))

for place, color in zip(places, colors):
    place_data = book_image_cks[book_image_cks['place'] == place]
    ax.scatter(
        place_data['year'],
        place_data['unique_images'],
        label=place,
        s=place_data['unique_images'] * 50,
        alpha=0.7,
        color=color,
        edgecolors='black',
        linewidth=1
    )

ax.set_xlabel('Publication Year', fontsize=12)
ax.set_ylabel('Number of Unique Images per Book', fontsize=12)
ax.set_title('Rising and Setting Books by Publication Place\nSize = Number of Images', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("\n=== Summary Statistics ===")
print(f"Total books with rising/setting images: {len(book_image_cks)}")
print(f"Year range: {book_image_cks['year'].min()} - {book_image_cks['year'].max()}")
print(f"\nImage count per book:")
print(book_image_cks['unique_images'].describe())
print(f"\nCKS types distribution:")
print(book_image_cks['num_cks_types'].value_counts().sort_index())
print(f"\nBooks by publication place:")
print(book_image_cks['place'].value_counts())